# Урок 7

## Генерация текста с помощью RNN

Поставим задачу: необходимо сгенерировать научный текст.

В качестве датасета возьмем [выгрузку статей из Arxiv](https://www.kaggle.com/datasets/Cornell-University/arxiv). Используем версию 175 (4.13 GB).

Метрику качества выберем "на глаз".

Модель возьмем RNN, функция потерь будет кросс-энтропия, оптимизатор Adam.

Модель будет предсказывать следующий токен. Кросс-энтропию будем считать над вероятностью реального токена.

In [1]:
from random import sample

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import tqdm
import wandb
from torch.optim import Adam

### Токенизация

В качестве токена в этой задаче будем брать **один символ**.

In [2]:
# BOS — символ начала текста, EOS — символ конца текста
BOS, EOS = " ", "\n"

lines = []
# возьмем только каждую 10-ую строку в финальный датасет
with open("/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json", "r") as f:
    for i, one_line in enumerate(tqdm.tqdm(f.readlines())):
        if i % 10 == 0:
            lines.append(one_line)
with open("small-data.json", "w") as f:
    f.writelines(lines)

data = pd.read_json("small-data.json", lines=True)
lines = (
    data.apply(lambda row: (row["title"] + " ; " + row["abstract"])[:512], axis=1)
    .apply(lambda line: BOS + line.replace(EOS, " ") + EOS)
    .tolist()
)

100%|██████████| 2982054/2982054 [00:01<00:00, 2762196.10it/s]


In [ ]:
lines[0]

In [5]:
tokens = {one_char for one_line in lines for one_char in one_line}

tokens = sorted(tokens)
n_tokens = len(tokens)
print("n_tokens = ", n_tokens)

n_tokens =  100


In [6]:
token_to_id = {x: i for i, x in enumerate(tokens)}
token_to_id

{'\n': 0,
 ' ': 1,
 '!': 2,
 '"': 3,
 '#': 4,
 '$': 5,
 '%': 6,
 '&': 7,
 "'": 8,
 '(': 9,
 ')': 10,
 '*': 11,
 '+': 12,
 ',': 13,
 '-': 14,
 '.': 15,
 '/': 16,
 '0': 17,
 '1': 18,
 '2': 19,
 '3': 20,
 '4': 21,
 '5': 22,
 '6': 23,
 '7': 24,
 '8': 25,
 '9': 26,
 ':': 27,
 ';': 28,
 '<': 29,
 '=': 30,
 '>': 31,
 '?': 32,
 '@': 33,
 'A': 34,
 'B': 35,
 'C': 36,
 'D': 37,
 'E': 38,
 'F': 39,
 'G': 40,
 'H': 41,
 'I': 42,
 'J': 43,
 'K': 44,
 'L': 45,
 'M': 46,
 'N': 47,
 'O': 48,
 'P': 49,
 'Q': 50,
 'R': 51,
 'S': 52,
 'T': 53,
 'U': 54,
 'V': 55,
 'W': 56,
 'X': 57,
 'Y': 58,
 'Z': 59,
 '[': 60,
 '\\': 61,
 ']': 62,
 '^': 63,
 '_': 64,
 '`': 65,
 'a': 66,
 'b': 67,
 'c': 68,
 'd': 69,
 'e': 70,
 'f': 71,
 'g': 72,
 'h': 73,
 'i': 74,
 'j': 75,
 'k': 76,
 'l': 77,
 'm': 78,
 'n': 79,
 'o': 80,
 'p': 81,
 'q': 82,
 'r': 83,
 's': 84,
 't': 85,
 'u': 86,
 'v': 87,
 'w': 88,
 'x': 89,
 'y': 90,
 'z': 91,
 '{': 92,
 '|': 93,
 '}': 94,
 '~': 95,
 '\x7f': 96,
 '\x80': 97,
 '\x99': 98,
 'â': 99}

### Паддинги

In [7]:
def to_tensor(
    lines: list[str],
    max_len: int | None = None,
    pad: str = token_to_id[EOS],
    dtype=torch.int64,
):
    max_len = max_len or max(map(len, lines))
    lines_ix = torch.full([len(lines), max_len], pad, dtype=dtype)
    for i in range(len(lines)):
        line_ix = [token_to_id[x] for x in lines[i][:max_len]]
        lines_ix[i, : len(line_ix)] = torch.tensor(line_ix)
    return lines_ix


print(to_tensor([" abc\n", " abacaba\n", " abc1234567890\n"], 15))

tensor([[ 1, 66, 67, 68,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 1, 66, 67, 66, 68, 66, 67, 66,  0,  0,  0,  0,  0,  0,  0],
        [ 1, 66, 67, 68, 18, 19, 20, 21, 22, 23, 24, 25, 26, 17,  0]])


In [8]:
def compute_mask(input_ix, eos_ix=token_to_id[EOS]):
    return F.pad(
        torch.cumsum(input_ix == eos_ix, dim=-1)[..., :-1] < 1,
        pad=(1, 0, 0, 0),
        value=True,
    )


print(compute_mask(to_tensor([" hello there\n"], max_len=15)))
print(compute_mask(to_tensor([" hello there\n"], max_len=18)))

tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True, False, False]])
tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True, False, False, False, False, False]])


### Модель

In [ ]:
ref_rnn = nn.RNN(10, 32)
print(ref_rnn.weight_hh_l0.shape)
print(ref_rnn.weight_ih_l0.shape)
print(ref_rnn.bias_hh_l0.shape)
print(ref_rnn.bias_ih_l0.shape)

In [9]:
class MyRnnLayer(nn.Module):
    # tanh(W_x * x + W_h * h + b_x + b_h) = tanh(W_x * x + b_x     +     W_h * h + b_h)
    def __init__(self, in_dim: int, hid_size: int):
        super().__init__()
        self.hid_size = hid_size
        self.input_linear_layer = nn.Linear(
            in_features=in_dim, out_features=hid_size, bias=True
        )
        self.hidden_state_linear_layer = nn.Linear(
            in_features=hid_size, out_features=hid_size, bias=True
        )
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor, state: torch.Tensor | None = None):
        assert x.ndim == 3  # (bs, n_tokens, token_dim)
        if state is None:
            state = torch.zeros((x.shape[0], self.hid_size), device=x.device)
        states = [state]
        for i in range(x.shape[1]):
            # print(states[-1].shape)
            states.append(
                self.activation(
                    self.input_linear_layer(x[:, i, :])
                    + self.hidden_state_linear_layer(states[-1])
                )
            )
        return torch.stack(states[1:]).permute((1, 0, 2)), states[-1][None, ...]


my_rnn = MyRnnLayer(10, 32)
test_input = torch.randn((5, 12, 10))
test_output = my_rnn(test_input)
test_output[0].shape, test_output[1].shape

(torch.Size([5, 12, 32]), torch.Size([1, 5, 32]))

In [ ]:
ref_rnn = nn.RNN(10, 32, batch_first=True)
ref_output = ref_rnn(test_input)
ref_output[0].shape, ref_output[1].shape

In [10]:
from torch.testing import assert_close

# Протестируем: наш слой должен давать те же результаты, что слой в pytorch
ref_rnn = nn.RNN(10, 32, batch_first=True)
with torch.no_grad():
    my_rnn.hidden_state_linear_layer.weight.copy_(ref_rnn.weight_hh_l0.clone())
    my_rnn.input_linear_layer.weight.copy_(ref_rnn.weight_ih_l0.clone())
    my_rnn.hidden_state_linear_layer.bias.copy_(ref_rnn.bias_hh_l0.clone())
    my_rnn.input_linear_layer.bias.copy_(ref_rnn.bias_ih_l0.clone())
    for i in (0, 1):
        actual = my_rnn(test_input)[i]
        expected = ref_rnn(test_input)[i]
        assert_close(actual, expected)

In [ ]:
# 3 токена
# [0, 1, 2]
# размерность эмбеддинга 2
# 3x2 — матрица эмбеддингов

"""
[
    [a, b],
    [c, d],
    [e, f],
]
"""

# embbeding[0] --> [a, b]
# embbeding[2] --> [e, f]

In [11]:
class RNNLanguageModel(nn.Module):
    def __init__(
        self, n_tokens: int = n_tokens, emb_size: int = 16, hid_size: int = 256
    ):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings=n_tokens, embedding_dim=emb_size)
        self.rnn = MyRnnLayer(emb_size, hid_size)
        # То же самое, что:
        # self.rnn = nn.RNN(emb_size, hid_size, batch_first=True)
        self.linear = nn.Linear(in_features=hid_size, out_features=n_tokens)

    def forward(self, input_ix):
        # input_ix -> (bs, n_tokens) = (1, 4)
        rv = self.emb(input_ix)  # (1, 4, 16)
        rv = self.rnn(rv)[0]  # (1, 4, 256)
        # здесь 97 - размер словаря, количество токенов для которых вычисляем вероятность
        rv = self.linear(rv)  # (1, 4, 256) @ (256, 97) = (1, 4, 97)
        return rv


model = RNNLanguageModel()
model(to_tensor(["in this article we"])).shape

torch.Size([1, 18, 100])

### Функция потерь и генерация текста

#### Функция потерь
Модель выдает вероятности каждого токена из словаря для каждого слова.
Будем учить модель предсказывать следующий токен при условии всех предыдущих.

![pic](https://raw.githubusercontent.com/AndreyKurdyubov/Karpov.courses_DL/9ac0258b42d944494da4e2a7d27b0863559c3349/Base_DL/RNN_example.png)

Формула функции потерь:
$$
L = - \cfrac{1}{N} \sum_{i=1}^N \ln p(x_t^{(i)} | x_{t-1}^{(i)}, \dots, x_1^{(i)})
$$
где $N$ — размер батча.

#### Генерация
Поскольку модель выдает вектор вероятностей каждого токена в предложении, то для генерации одного текста поступим так:
1. Берем вектор `(batch_size = 1, n_words, emb_dim)` из выхода модели, берем последнюю координату из `n_words`.
2. Из полученного вектора `(1, emb_dim)` вероятностей решаем, как получить следующий токен.

Существует два варианта того, как можно получить следующий токен:
1. Отобрать тот, у кого самая большая вероятность — это **жадный выбор** (greedy sampling). Просто берем `argmax`.
2. Взять случайный с учетом вероятностей.
Модель выдает логиты — к ним применим softmax и получим вероятности.
Пример сэмплирования: в векторе `[0.1, 0.3, 0.6]` первая координата будет выбрана с вероятностью 10%, вторая — с 30%, третья — с 60%.
Это называется **случайное сэмплирование с учетом вероятностей**.
3. Взять идею из п.2, но перед сэмплированием перевзвесить все логиты:
$$
p(l_i) = \cfrac{\exp(l_i / \tau)}{\sum_{i=1}^D \exp(l_i / \tau)}
$$
где $D$ — мощность словаря, число уникальных токенов, $l_i$ — логит для $i$-го токена, $\tau$ — некоторый параметр, называемый **температурой**.

Это называется **сэмплирование с температурой**.
Варьируя температуру, можно либо перейти в жадный выбор, либо в равновероятный выбор токенов.

Мы будем использовать п.1 и п.3 для сэмплирования.

In [ ]:
# пример как работает torch.gather
# тенхор, откуда забираем значения
rv = torch.tensor([
    [[0, 1]], 
    [[4, 5]], 
    [[8, 9]]
])

# тенхор с индексами элементов, которые нужно забрвть
reference_answers = torch.tensor([
    [0], 
    [1], 
    [0]
])
print(rv.shape, reference_answers.shape)
print(reference_answers[:, :, None])

torch.gather(rv, 2, reference_answers[:, :, None]).squeeze(2)

In [12]:
def compute_loss(model: nn.Module, input_ix: torch.Tensor, device: str = "cpu"):
    input_ix = torch.as_tensor(input_ix, dtype=torch.int64)
    input_ix = input_ix.to(device)

    # (bs, sentence_length)
    logits = model(input_ix[:, :-1])  # (bs, sentence_length, 97)
    reference_answers = input_ix[:, 1:]  # (bs, sentence_length) [32, 13, 28, 1]
    rv = torch.softmax(logits, 2)    # (bs, sentence_length, 97) --> (bs, sentence_length)
    # (bs, sentence_length, 97)
    # (bs, [0], [32])
    # (bs, [1], [13])
    # (bs, [2], [28])
    # (bs, [3], [1])
    # [:, :, 0]  # (a, b, c) -> (a, b)
    rv = torch.gather(rv, 2, reference_answers[:, :, None]).squeeze(2)  # (bs, sentence_length)
    rv = torch.log(rv)
    # Потери считаем только для реальных токенов, паддинги не включаем.
    # Для этого умножим на маску — она равна 1 для реальных токенов и 0 для паддингов.
    rv = rv * compute_mask(input_ix)[:, 1:]
    return -torch.sum(rv) / input_ix.shape[0]


def generate(
    model: nn.Module,
    prefix: str = BOS,
    temperature: float = 1.0,
    max_len: int = 100,
    device: str = "cpu",
):
    with torch.no_grad():
        while True:
            # неправильное применение температуры автором
            # probs = (
            #     torch.softmax(model(to_tensor([prefix]).to(device))[0, -1], dim=-1)
            #     .cpu()
            #     .numpy()
            # )
            # if temperature == 0:
            #     next_token = tokens[np.argmax(probs)]
            # else:
            #     probs = np.array([p ** (1.0 / temperature) for p in probs])
            #     probs /= sum(probs)
            #     next_token = np.random.choice(tokens, p=probs)

            logits = model(to_tensor([prefix]).to(device))[0, -1]  # Получаем логиты

            if temperature == 0:
                # Жадная декодировка
                next_token = tokens[torch.argmax(logits, dim=-1).item()]
            else:
                # Правильный temperature scaling: логиты / температуру -> softmax
                scaled_logits = logits / temperature
                probs = torch.softmax(scaled_logits, dim=-1).cpu().numpy()
                next_token = np.random.choice(tokens, p=probs)

            prefix += next_token
            if next_token == EOS or len(prefix) > max_len:
                break
    return prefix

### Обучение и результаты

In [16]:
def train_loop(model: nn.Module, train_lines: list[str], device: str = "cpu"):
    run = wandb.init(project="start-dl--lesson-7")
    clip_norm = 1e5
    batch_size = 64
    opt = Adam(model.parameters())
    train_history = []
    model.to(device)
    text_table = wandb.Table(columns=["iteration", "text"])
    for i in tqdm.trange(len(train_history), 1000):
        batch = to_tensor(sample(train_lines, batch_size)).to(device)
        loss_i = compute_loss(model, batch, device=device)

        opt.zero_grad()
        loss_i.backward()
        # При взрыве сети из-за расхождения градиента ,можно его ограничить
        # nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        opt.step()

        train_history.append((i, float(loss_i)))
        wandb.log({"loss": loss_i.detach().cpu().item()})

        if (i + 1) % 50 == 0:
            for _ in range(3):
                example = generate(model, temperature=0.5, device=device)
                text_table.add_data(i, example)
    run.log({"train-samples": text_table})


# model = RNNLanguageModel()
device = "cuda" if torch.cuda.is_available() else "cpu"
train_loop(model, lines, device)

loss,█▇▇▇▆▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▂▁▁
loss,760.04388


100%|██████████| 1000/1000 [03:47<00:00,  4.39it/s]


In [19]:
for _ in range(5):
    print('###')
    print(generate(model, temperature=0.5, max_len=200, device=device))

###
 Computation ;   We study the conditions on the problem in a quantitative Language Model ;   We present a real and a nonlown the present an assistence of the graph then have been study energy post and 
###
 On the continuous computations of the recent the promising the action of the light strong and the semiconductor provides the interaction of the lognities of the interactions of problem and the interse
###
 Anomaloge is a specifically interactions ;   We study the Process consider the large finite state and the quantum methods are consider the results for complex to the also and property control of the c
###
 Context method for the use of consistent superconductivity in the state of interaction of an integral properties of the set of the states and information and the sensor using the resolation of the lar
###
 General states and the resolved by a semiential structure of problem and the interaction ;   We study the obtained by a results in the contain model for the dimension of t

In [20]:
for _ in range(5):
    print('###')
    print(generate(model, prefix=" Exciton-polaritons are", temperature=1, max_len=200, device=device))

###
 Exciton-polaritons are netweing modal Summay integrable found, with the problem of quantum properties into an understematical model in the symme

###
 Exciton-polaritons are matter. The mass, ak elements is Nitwicle distspluds introduce, the resulting algebras ; Nege notion to the time. We present remous" use of the revistence-belew-performances to 
###
 Exciton-polaritons are a periodul (CIBS. The challenge of higher correspondence are twarmonic functions of the electrons codes and molots to explores. In V$Tuhe is method that an Inbard expected is an
###
 Exciton-polaritons are severatil relay desigh infurity function lew for sequency decoduint model weirand A JThoovires ,naks Pland world model of the Luili-equatories ;   Fraysic A churanchien onzirate
###
 Exciton-polaritons are structure 



In [29]:
for _ in range(5):
    print('###')
    print(generate(model, prefix=" Exciting", temperature=1, max_len=200, device=device))

###
 Exciting reason underlysive reduce is a special glusting and exactling-2neveher to the ch

###
 Excitingation $\D key for downlly-types of properties and high-user (td, \7) matter observable possibul of WinI,. A. hard. be enholic parifying are studes, while those $n+t-3-lAPC   Cantical authorrop
###
 Exciting findrings this postable maj chealing the linear sqrating Fiq V\signaliza}^{j,\mol/s}T_2$ ; A local networks of the the out an altermate the forre   Lets in shell objective action $S$-a   with
###
 Exciting a images on some and suborgeniz paperthered Distance of Recentures with emergery suffers a way. Dequition, expected more spectra for to constraces show to constructed as a Uustribe momentum s
###
 Exciting approach Emotion A Du chalaringer and is vetrynicality gravity, the Languaks of MynCturnan \2925 has been a relations, the Collention:    chinci



In [23]:
# torch.save(model.state_dict(), "model-RNN_lect7.pt")

Получилось вполне правдоподобно.

В качестве упражнения попробуйте подвигать температуру:
1. Слишком большая температура сделает токены более равновероятными. Из-за этого слова начнут превращаться в кашу.
2. Слишком маленькая температура будет делать обратное — заострять пики распределения. Из-за этого сэмплирование превратится в жадное.

In [ ]:
res = model.cpu()
prefix = " Congatulations"
result = res(to_tensor([prefix]))

## Резюме

1. Узнали, как решать задачу генерации текста с помощью RNN в PyTorch.
2. Посмотрели, как формализовать задачу предсказания токена.
3. Познакомились со стратегиями генерации токенов:
    - жадное сэмплирование;
    - случайное сэмплирование с температурой.